## EDA — TradeMaster Pro
All 12 datasets: BTC / ETH / SOL × 1h / 4h / 12h / 1d

1. Structural integrity
2. Missing values
3. Descriptive statistics + sanity bounds
4. Infinite values
5. Target distribution — class balance
6. Multicollinearity — correlation matrix
7. Stationarity check
8. Outlier detection on returns / volume
---
**TradeMaster Pro — Platform-Specific EDA**
- TM-1. Win Rate vs Promotion Thresholds
- TM-2. Signal Quality by Session & Regime
- TM-3. Risk-to-Reward Distribution
- TM-4. Automatic Signal Risk Score
- TM-5. Multi-Asset × Multi-Timeframe Win Rate Matrix

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller


In [2]:
ASSETS     = ["BTC", "ETH", "SOL"]
TIMEFRAMES = ["1h", "4h", "12h", "1d"]
TARGET_DIR = {"1h": "target_dir_1h",  "4h": "target_dir_4h",
              "12h": "target_dir_12h", "1d": "target_dir_24h"}
TARGET_RET = {"1h": "target_ret_1h",  "4h": "target_ret_4h",
              "12h": "target_ret_12h", "1d": "target_ret_24h"}
# 3-month rolling window in bars per timeframe
ROLL_WIN   = {"1h": 2160, "4h": 540, "12h": 180, "1d": 90}

datasets = {}
for asset in ASSETS:
    for tf in TIMEFRAMES:
        key  = f"{asset}_{tf}"
        path = f"datasets/historical/{asset}_USDT_{tf}.csv"
        d    = pd.read_csv(path, parse_dates=["timestamp"]).set_index("timestamp")
        datasets[key] = d
        print(f"  {key:12s}  shape={d.shape}  {d.index.min().date()} to {d.index.max().date()}")

print(f"\nLoaded {len(datasets)} datasets.")


  BTC_1h        shape=(77684, 102)  2017-08-17 to 2026-07-03
  BTC_4h        shape=(19444, 102)  2017-08-17 to 2026-07-04
  BTC_12h       shape=(6486, 102)  2017-08-17 to 2026-07-04
  BTC_1d        shape=(3244, 102)  2017-08-17 to 2026-07-04
  ETH_1h        shape=(77684, 102)  2017-08-17 to 2026-07-03
  ETH_4h        shape=(19444, 102)  2017-08-17 to 2026-07-04
  ETH_12h       shape=(6486, 102)  2017-08-17 to 2026-07-04
  ETH_1d        shape=(3244, 102)  2017-08-17 to 2026-07-04
  SOL_1h        shape=(50892, 102)  2020-09-11 to 2026-07-03
  SOL_4h        shape=(12735, 102)  2020-09-11 to 2026-07-04
  SOL_12h       shape=(4245, 102)  2020-09-11 to 2026-07-04
  SOL_1d        shape=(2123, 102)  2020-09-11 to 2026-07-04

Loaded 12 datasets.


### 1. Structural Integrity

In [3]:
FREQ_MAP = {"1h": "h", "4h": "4h", "12h": "12h", "1d": "D"}

rows = []
for asset in ASSETS:
    for tf in TIMEFRAMES:
        key = f"{asset}_{tf}"
        d   = datasets[key]
        expected = pd.date_range(d.index.min(), d.index.max(), freq=FREQ_MAP[tf])
        missing  = expected.difference(d.index)
        rows.append({
            "Dataset":      key,
            "Rows":         len(d),
            "Cols":         d.shape[1],
            "Start":        str(d.index.min().date()),
            "End":          str(d.index.max().date()),
            "Missing bars": len(missing),
            "Dup index":    int(d.index.duplicated().sum()),
        })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))


Dataset  Rows  Cols      Start        End  Missing bars  Dup index
 BTC_1h 77684   102 2017-08-17 2026-07-03           128          0
 BTC_4h 19444   102 2017-08-17 2026-07-04            16          0
BTC_12h  6486   102 2017-08-17 2026-07-04             1          0
 BTC_1d  3244   102 2017-08-17 2026-07-04             0          0
 ETH_1h 77684   102 2017-08-17 2026-07-03           128          0
 ETH_4h 19444   102 2017-08-17 2026-07-04            16          0
ETH_12h  6486   102 2017-08-17 2026-07-04             1          0
 ETH_1d  3244   102 2017-08-17 2026-07-04             0          0
 SOL_1h 50892   102 2020-09-11 2026-07-03            20          0
 SOL_4h 12735   102 2020-09-11 2026-07-04             0          0
SOL_12h  4245   102 2020-09-11 2026-07-04             0          0
 SOL_1d  2123   102 2020-09-11 2026-07-04             0          0


### 2. Missing Values

In [4]:
# Total NaN count per dataset
nan_totals = {}
for key, d in datasets.items():
    nan_totals[key] = d.isna().sum()

nan_df = pd.DataFrame(nan_totals).T          # rows=datasets, cols=features
nan_df = nan_df.loc[:, (nan_df > 0).any()]   # keep only cols with any NaN

print(f"Features with NaN in at least one dataset: {nan_df.shape[1]}")
print("\nNaN count per dataset (non-zero columns only):")
print(nan_df.to_string())

# Heatmap — clip at 200 for colour clarity
fig, ax = plt.subplots(figsize=(20, 6))
sns.heatmap(nan_df.clip(upper=200), cmap="YlOrRd", ax=ax,
            linewidths=0.3, cbar_kws={"label": "NaN count (capped 200)"})
ax.set_title("Missing Values Heatmap — All 12 Datasets", fontsize=13)
ax.tick_params(axis="x", rotation=90, labelsize=6)
ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.savefig("eda_missing_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nSaved: eda_missing_heatmap.png")


Features with NaN in at least one dataset: 69

NaN count per dataset (non-zero columns only):
         adx  bb_lower  bb_mid  bb_pct  bb_upper  bb_width   cci  dist_resist  dist_support  ema200_slope  ema50_slope  gap_pct  high_100  kijun  low_100  minus_di  obv_ma20  pct_from_high  pct_from_low  plus_di  resist_50  ret_1  ret_12  ret_24  ret_3  ret_48  ret_6  rsi_14  rsi_21  rsi_7  senkou_a  senkou_b  sma_20  sma_50  sr_width  stoch_d  stoch_k  support_50  target_dir_12h  target_dir_144h  target_dir_16h  target_dir_1h  target_dir_24h  target_dir_288h  target_dir_48h  target_dir_4h  target_dir_576h  target_dir_96h  target_ret_12h  target_ret_144h  target_ret_16h  target_ret_1h  target_ret_24h  target_ret_288h  target_ret_48h  target_ret_4h  target_ret_576h  target_ret_96h  tenkan  trend_consistency  trend_strength  vol_ma20  vol_ratio  vol_regime_20  vol_regime_ratio  vol_stddev  vs_vwap  vwap_24  willr
BTC_1h   1.0      19.0    19.0    19.0      19.0      19.0  19.0         49.0      

### 3. Descriptive Statistics + Sanity Bounds

In [5]:
rows = []
for key, d in datasets.items():
    # RSI_14 bounds [0, 100]
    rsi = d["rsi_14"].dropna()
    rsi_invalid = int((~rsi.between(0, 100)).sum())

    # BB_pct bounds [-5, 5]
    bb = d["bb_pct"].dropna() if "bb_pct" in d.columns else pd.Series(dtype=float)
    bb_invalid = int((~bb.between(-5, 5)).sum()) if len(bb) else 0

    # ATR must be > 0
    atr = d["atr_14"].dropna()
    atr_invalid = int((atr <= 0).sum())

    rows.append({
        "Dataset":       key,
        "RSI invalid":   rsi_invalid,
        "BB_pct invalid":bb_invalid,
        "ATR<=0":        atr_invalid,
        "close min":     round(d["close"].min(), 2),
        "close max":     round(d["close"].max(), 2),
        "ret_1 min":     round(d["ret_1"].min(), 4) if "ret_1" in d.columns else None,
        "ret_1 max":     round(d["ret_1"].max(), 4) if "ret_1" in d.columns else None,
    })

print(pd.DataFrame(rows).to_string(index=False))


Dataset  RSI invalid  BB_pct invalid  ATR<=0  close min  close max  ret_1 min  ret_1 max
 BTC_1h            0               0       0    2919.00  126011.18    -0.2010     0.1603
 BTC_4h            0               0       0    2919.00  125410.81    -0.2294     0.2716
BTC_12h            0               0       0    2919.00  124658.54    -0.2684     0.2371
 BTC_1d            0               0       0    3189.02  124658.54    -0.5026     0.2030
 ETH_1h            0               0       0      82.17    4935.00    -0.2341     0.1662
 ETH_4h            0               0       0      82.17    4846.71    -0.2495     0.2747
ETH_12h            0               0       0      83.67    4832.07    -0.3540     0.2387
 ETH_1d            0               0       0      83.76    4832.07    -0.5905     0.2338
 SOL_1h            0               0       0       1.18     286.24    -0.2142     0.1710
 SOL_4h            0               0       0       1.20     286.24    -0.2213     0.2686
SOL_12h            0 

### 4. Infinite Values

In [6]:
rows = []
for key, d in datasets.items():
    num    = d.select_dtypes(include=[np.number])
    n_inf  = int(np.isinf(num).sum().sum())
    rows.append({"Dataset": key, "Inf count": n_inf,
                 "Status": "Clean" if n_inf == 0 else "Has Inf"})

inf_df = pd.DataFrame(rows)
print(inf_df.to_string(index=False))
print(f"\nDatasets with Inf: {(inf_df['Inf count'] > 0).sum()}")


Dataset  Inf count Status
 BTC_1h          0  Clean
 BTC_4h          0  Clean
BTC_12h          0  Clean
 BTC_1d          0  Clean
 ETH_1h          0  Clean
 ETH_4h          0  Clean
ETH_12h          0  Clean
 ETH_1d          0  Clean
 SOL_1h          0  Clean
 SOL_4h          0  Clean
SOL_12h          0  Clean
 SOL_1d          0  Clean

Datasets with Inf: 0


### 5. Target Distribution — Class Balance

In [7]:
fig, axes = plt.subplots(3, 4, figsize=(20, 13))
class_colors = {0.0: "#e74c3c", 1.0: "#95a5a6", 2.0: "#2ecc71"}
label_map    = {0.0: "Down", 1.0: "Flat", 2.0: "Up"}

for r, asset in enumerate(ASSETS):
    for c, tf in enumerate(TIMEFRAMES):
        ax  = axes[r][c]
        key = f"{asset}_{tf}"
        d   = datasets[key]
        tc  = TARGET_DIR[tf]
        counts = d[tc].value_counts(normalize=True, dropna=True).sort_index()
        bars = ax.bar(
            [label_map.get(k, str(k)) for k in counts.index],
            counts.values * 100,
            color=[class_colors.get(k, "steelblue") for k in counts.index],
            edgecolor="none"
        )
        ax.axhline(33.3, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
        ax.set_ylim(0, 80)
        ax.set_title(f"{key}", fontsize=10, fontweight="bold")
        ax.set_ylabel("% of bars" if c == 0 else "")
        ax.tick_params(labelsize=8)
        for bar, val in zip(bars, counts.values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f"{val*100:.1f}%", ha="center", va="bottom", fontsize=7)

plt.suptitle("Target Direction Class Balance — All 12 Datasets", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("eda_target_dist.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: eda_target_dist.png")

# Imbalance summary table
rows = []
for asset in ASSETS:
    for tf in TIMEFRAMES:
        key = f"{asset}_{tf}"
        d   = datasets[key]
        tc  = TARGET_DIR[tf]
        vc  = d[tc].value_counts(normalize=True, dropna=True)
        rows.append({
            "Dataset": key,
            "Down %":  round(vc.get(0.0, 0)*100, 1),
            "Flat %":  round(vc.get(1.0, 0)*100, 1),
            "Up %":    round(vc.get(2.0, 0)*100, 1),
            "Max class %": round(vc.max()*100, 1),
        })

print(pd.DataFrame(rows).to_string(index=False))


Saved: eda_target_dist.png
Dataset  Down %  Flat %  Up %  Max class %
 BTC_1h    13.1    73.2  13.7         73.2
 BTC_4h    24.4    49.6  26.0         49.6
BTC_12h    32.7    31.9  35.4         35.4
 BTC_1d    38.3    20.7  41.0         41.0
 ETH_1h    17.9    63.2  18.8         63.2
 ETH_4h    29.7    38.9  31.4         38.9
ETH_12h    37.6    22.9  39.5         39.5
 ETH_1d    41.9    15.2  43.0         43.0
 SOL_1h    25.5    49.0  25.5         49.0
 SOL_4h    36.8    26.3  36.9         36.9
SOL_12h    42.7    15.1  42.2         42.7
 SOL_1d    45.6     9.9  44.4         45.6


### 6. Multicollinearity — Correlation Matrix
One heatmap per asset (using 1h data). Features are identical across timeframes; 1h gives maximum sample size.

In [8]:
feature_groups = {
    "Price/Returns":   ["ret_1","ret_3","ret_6","ret_12","ret_24","ret_48","gap_pct"],
    "Momentum":        ["rsi_7","rsi_14","rsi_21","macd","macd_signal","macd_hist",
                        "stoch_k","stoch_d","willr","cci","momentum"],
    "Trend":           ["adx","plus_di","minus_di","ema50_slope","ema200_slope",
                        "trend_strength","trend_consistency",
                        "ema_cross_9_21","ema_cross_21_50","ema_cross_50_200"],
    "Volatility":      ["atr_14","atr_21","atr_ratio","bb_width","bb_pct",
                        "vol_ratio","vol_stddev","vol_regime_20","vol_regime_ratio","is_high_vol"],
    "Ichimoku":        ["tenkan","kijun","senkou_a","senkou_b","cloud_bull","tenkan_kijun_cross"],
    "S/R & Price Pos": ["pct_from_high","pct_from_low","dist_resist","dist_support","sr_width",
                        "vs_ema_9","vs_ema_21","vs_ema_50","vs_ema_100","vs_ema_200","vs_vwap",
                        "dd_from_ath"],
}

for asset in ASSETS:
    d = datasets[f"{asset}_1h"]
    fig, axes = plt.subplots(2, 3, figsize=(22, 14))
    axes = axes.flatten()
    for ax, (grp, cols) in zip(axes, feature_groups.items()):
        valid = [c for c in cols if c in d.columns]
        corr  = d[valid].corr()
        mask  = np.triu(np.ones_like(corr, dtype=bool))
        sns.heatmap(corr, mask=mask, ax=ax, cmap="RdBu_r", center=0,
                    vmin=-1, vmax=1, annot=len(valid) <= 8, fmt=".2f",
                    linewidths=0.3, xticklabels=True, yticklabels=True)
        ax.set_title(grp, fontsize=11, fontweight="bold")
        ax.tick_params(axis="x", rotation=45, labelsize=7)
        ax.tick_params(axis="y", labelsize=7)
    plt.suptitle(f"{asset} 1h — Feature Correlation Heatmaps", fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    fname = f"eda_corr_{asset}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved: {fname}")

# High-corr pairs across all assets (1h)
print("\n=== Highly correlated pairs (|r| > 0.90) per asset ===")
all_feats = [c for g in feature_groups.values() for c in g]
for asset in ASSETS:
    d     = datasets[f"{asset}_1h"]
    feats = [c for c in all_feats if c in d.columns]
    corr  = d[feats].corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    pairs = upper.stack().reset_index()
    pairs.columns = ["A", "B", "r"]
    pairs = pairs[pairs["r"] > 0.90].sort_values("r", ascending=False)
    print(f"  {asset}: {len(pairs)} pairs")
    if len(pairs):
        print(pairs.head(10).to_string(index=False))


Saved: eda_corr_BTC.png
Saved: eda_corr_ETH.png
Saved: eda_corr_SOL.png

=== Highly correlated pairs (|r| > 0.90) per asset ===
  BTC: 25 pairs
          A         B        r
    stoch_k     willr 1.000000
     tenkan     kijun 0.999880
   senkou_a  senkou_b 0.999792
      kijun  senkou_a 0.999533
      kijun  senkou_b 0.999199
     tenkan  senkou_a 0.999174
     tenkan  senkou_b 0.998855
     atr_14    atr_21 0.995386
     rsi_14    rsi_21 0.978149
ema50_slope vs_ema_50 0.976157
  ETH: 25 pairs
          A         B        r
    stoch_k     willr 1.000000
     tenkan     kijun 0.999692
   senkou_a  senkou_b 0.999453
      kijun  senkou_a 0.998763
      kijun  senkou_b 0.997889
     tenkan  senkou_a 0.997817
     tenkan  senkou_b 0.996989
     atr_14    atr_21 0.995954
     rsi_14    rsi_21 0.978090
ema50_slope vs_ema_50 0.977184
  SOL: 29 pairs
       A        B        r
 stoch_k    willr 1.000000
  tenkan    kijun 0.999506
senkou_a senkou_b 0.999130
   kijun senkou_a 0.998141
   kiju

### 7. Stationarity Check
ADF test on key features for all 12 datasets (subsampled to 5 000 rows for speed).

In [9]:
test_feats = ["close","ret_1","rsi_14","macd","bb_pct","atr_14","vs_ema_50","vol_ratio","adx"]

rows = []
for key, d in datasets.items():
    for feat in test_feats:
        if feat not in d.columns:
            continue
        series = d[feat].dropna()
        series = series.iloc[::max(1, len(series)//5000)]   # subsample
        stat, p = adfuller(series, autolag="AIC")[:2]
        rows.append({"Dataset": key, "Feature": feat,
                     "ADF": round(stat, 3), "p": round(p, 4),
                     "Stationary": "Yes" if p < 0.05 else "No"})

stat_df = pd.DataFrame(rows)

# Pivot: rows=features, cols=datasets, values=Yes/No
pivot = stat_df.pivot(index="Feature", columns="Dataset", values="Stationary")
print(pivot.to_string())

non_stat = stat_df[stat_df["Stationary"] == "No"]
print(f"\nNon-stationary feature-dataset pairs: {len(non_stat)}")
if len(non_stat):
    print(non_stat[["Dataset","Feature","p"]].to_string(index=False))


Dataset   BTC_12h BTC_1d BTC_1h BTC_4h ETH_12h ETH_1d ETH_1h ETH_4h SOL_12h SOL_1d SOL_1h SOL_4h
Feature                                                                                         
adx           Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes
atr_14        Yes     No    Yes    Yes     Yes    Yes    Yes    Yes     Yes     No    Yes    Yes
bb_pct        Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes
close          No     No     No     No      No     No     No     No      No     No     No     No
macd          Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes
ret_1         Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes
rsi_14        Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes
vol_ratio     Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes     Yes    Yes    Yes    Yes
vs_ema_50     Yes    Yes    Ye

### 8. Outlier Detection on Returns & Volume

In [10]:
check_cols = ["ret_1", "ret_3", "ret_6", "volume", "vol_ratio", "atr_14"]

# Summary table
rows = []
for key, d in datasets.items():
    row = {"Dataset": key}
    for col in check_cols:
        if col not in d.columns:
            row[col] = None; continue
        s = d[col].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr     = q3 - q1
        n_out   = int(((s < q1 - 3*iqr) | (s > q3 + 3*iqr)).sum())
        row[col] = f"{n_out/len(s)*100:.2f}%"
    rows.append(row)

out_df = pd.DataFrame(rows)
print("Outlier rate (3×IQR) per dataset:")
print(out_df.to_string(index=False))

# Charts: one row per asset, one col per timeframe
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
for r, asset in enumerate(ASSETS):
    for c, tf in enumerate(TIMEFRAMES):
        ax  = axes[r][c]
        key = f"{asset}_{tf}"
        d   = datasets[key]
        col = "ret_1"
        s   = d[col].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1; lo = q1-3*iqr; hi = q3+3*iqr
        n_out = int(((s < lo) | (s > hi)).sum())
        ax.hist(s, bins=80, color="steelblue", edgecolor="none", alpha=0.8)
        ax.axvline(lo, color="red",    linestyle="--", linewidth=1)
        ax.axvline(hi, color="crimson",linestyle="--", linewidth=1)
        ax.set_title(f"{key}\noutliers {n_out/len(s)*100:.2f}%", fontsize=8)
        ax.tick_params(labelsize=7)

plt.suptitle("ret_1 Distribution + 3×IQR Outlier Bounds — All 12 Datasets",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("eda_outliers.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nSaved: eda_outliers.png")


Outlier rate (3×IQR) per dataset:
Dataset ret_1 ret_3 ret_6 volume vol_ratio atr_14
 BTC_1h 3.56% 3.65% 3.31%  5.50%     2.24%  0.24%
 BTC_4h 3.45% 2.67% 1.89%  5.45%     1.29%  0.03%
BTC_12h 2.84% 1.33% 0.86%  5.69%     1.21%  0.00%
 BTC_1d 1.82% 0.68% 0.90%  6.04%     1.27%  0.00%
 ETH_1h 2.91% 2.77% 2.38%  2.84%     2.31%  0.40%
 ETH_4h 2.62% 1.89% 1.33%  1.97%     1.40%  0.28%
ETH_12h 1.94% 1.16% 0.85%  1.70%     1.13%  0.23%
 ETH_1d 1.23% 0.77% 0.90%  1.60%     1.24%  0.22%
 SOL_1h 1.92% 1.88% 1.53%  3.01%     1.96%  0.46%
 SOL_4h 1.66% 1.16% 0.95%  2.47%     1.34%  0.35%
SOL_12h 1.25% 0.97% 0.71%  2.36%     1.59%  0.14%
 SOL_1d 0.99% 0.57% 0.76%  2.26%     1.47%  0.00%

Saved: eda_outliers.png


---
## TradeMaster Pro — Platform-Specific EDA
Directly maps dataset features to signal quality, win-rate thresholds, and risk scoring.

### TM-1. Win Rate vs Promotion Thresholds
Rolling 3-month win rate for all 12 datasets plotted against the platform's 60% / 70% / 75% thresholds.

In [11]:
fig, axes = plt.subplots(3, 4, figsize=(22, 13))

summary_rows = []
for r, asset in enumerate(ASSETS):
    for c, tf in enumerate(TIMEFRAMES):
        ax  = axes[r][c]
        key = f"{asset}_{tf}"
        d   = datasets[key]
        tc  = TARGET_DIR[tf]
        sub = d[d[tc].notna()].copy()
        sub["tp1_hit"] = (sub[tc] == 2).astype(int)

        wr_overall = sub["tp1_hit"].mean()
        rolling    = sub["tp1_hit"].rolling(ROLL_WIN[tf], min_periods=ROLL_WIN[tf]//4).mean()

        ax.plot(sub.index, rolling * 100, color="royalblue", linewidth=0.7, alpha=0.9)
        ax.axhline(60, color="orange", linestyle="--", linewidth=1,   label="60%")
        ax.axhline(70, color="green",  linestyle="--", linewidth=1,   label="70%")
        ax.axhline(75, color="red",    linestyle="--", linewidth=1,   label="75%")
        ax.set_ylim(20, 90)
        ax.set_title(f"{key}  (overall={wr_overall*100:.1f}%)", fontsize=8, fontweight="bold")
        ax.tick_params(labelsize=6)
        if r == 0 and c == 0:
            ax.legend(fontsize=6)

        above60 = (rolling >= 0.60).mean() * 100
        above75 = (rolling >= 0.75).mean() * 100
        summary_rows.append({"Dataset": key, "Overall WR%": round(wr_overall*100,2),
                              "% time >60%": round(above60,1), "% time >75%": round(above75,1)})

plt.suptitle("Rolling 3-Month Win Rate vs TradeMaster Promotion Thresholds", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("tm_win_rate.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: tm_win_rate.png\n")
print(pd.DataFrame(summary_rows).to_string(index=False))


Saved: tm_win_rate.png

Dataset  Overall WR%  % time >60%  % time >75%
 BTC_1h        13.72          0.0          0.0
 BTC_4h        26.01          0.0          0.0
BTC_12h        35.44          0.0          0.0
 BTC_1d        40.98          0.9          0.0
 ETH_1h        18.83          0.0          0.0
 ETH_4h        31.40          0.0          0.0
ETH_12h        39.52          0.0          0.0
 ETH_1d        42.95          0.1          0.0
 SOL_1h        25.54          0.0          0.0
 SOL_4h        36.89          0.0          0.0
SOL_12h        42.18          0.0          0.0
 SOL_1d        44.44          0.1          0.0


### TM-2. Signal Quality by Session & Volatility Regime
Win rate breakdown by Asia / EU / US session and High / Low vol for all 12 datasets.

In [12]:
rows = []
for asset in ASSETS:
    for tf in TIMEFRAMES:
        key = f"{asset}_{tf}"
        d   = datasets[key]
        tc  = TARGET_DIR[tf]
        sub = d[d[tc].notna()].copy()
        sub["tp1_hit"] = (sub[tc] == 2).astype(int)

        for sess, col in [("Asia","session_asia"),("EU","session_eu"),("US","session_us")]:
            if col not in sub.columns: continue
            grp = sub[sub[col]==1]["tp1_hit"]
            rows.append({"Dataset":key,"Session":sess,
                         "WR%": round(grp.mean()*100,2), "n": len(grp)})

        for regime, label in [(0,"Low Vol"),(1,"High Vol")]:
            grp = sub[sub["is_high_vol"]==regime]["tp1_hit"]
            rows.append({"Dataset":key,"Session":label,
                         "WR%": round(grp.mean()*100,2), "n": len(grp)})

sess_df = pd.DataFrame(rows)

# Pivot for heatmap: rows=Dataset, cols=Session
pivot = sess_df.pivot(index="Dataset", columns="Session", values="WR%")
fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn", center=50,
            linewidths=0.5, ax=ax, cbar_kws={"label": "Win Rate (%)"})
ax.set_title("Win Rate (%) by Session & Vol Regime — All 12 Datasets", fontsize=12)
plt.tight_layout()
plt.savefig("tm_session_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: tm_session_heatmap.png\n")
print(pivot.to_string())

# Hour x DayOfWeek heatmap for BTC_1h as reference
d1h = datasets["BTC_1h"]
sub = d1h[d1h[TARGET_DIR["1h"]].notna()].copy()
sub["tp1_hit"] = (sub[TARGET_DIR["1h"]] == 2).astype(int)
days = ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"]
pvt  = sub.pivot_table(values="tp1_hit", index="hour", columns="day_of_week", aggfunc="mean")
pvt.columns = days
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(pvt*100, cmap="RdYlGn", center=50, ax=ax,
            annot=True, fmt=".0f", linewidths=0.3, cbar_kws={"label": "Win Rate (%)"})
ax.set_title("BTC 1h — Win Rate % by Hour × Day of Week", fontsize=12)
plt.tight_layout()
plt.savefig("tm_hour_dow_heatmap.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: tm_hour_dow_heatmap.png")


Saved: tm_session_heatmap.png

Session   Asia     EU  High Vol  Low Vol     US
Dataset                                        
BTC_12h  37.50  33.37     39.13    34.96    NaN
BTC_1d   40.98    NaN     45.64    40.53    NaN
BTC_1h   11.59  14.95     18.49    13.02  15.59
BTC_4h   23.73  27.93     32.97    25.11  26.37
ETH_12h  40.67  38.37     44.05    38.95    NaN
ETH_1d   42.95    NaN     44.74    42.79    NaN
ETH_1h   16.65  19.93     25.17    17.95  20.61
ETH_4h   29.24  32.87     36.18    30.75  32.09
SOL_12h  40.81  43.54     46.06    41.70    NaN
SOL_1d   44.44    NaN     44.67    44.42    NaN
SOL_1h   24.82  25.96     31.44    24.86  26.89
SOL_4h   35.92  37.23     41.14    36.42  37.54
Saved: tm_hour_dow_heatmap.png


### TM-3. Risk-to-Reward (R:R) Distribution
ATR-based SL proxy vs actual forward return for all 12 datasets.

In [13]:
fig, axes = plt.subplots(3, 4, figsize=(20, 12))
rr_rows = []

for r, asset in enumerate(ASSETS):
    for c, tf in enumerate(TIMEFRAMES):
        ax  = axes[r][c]
        key = f"{asset}_{tf}"
        d   = datasets[key]
        tr  = TARGET_RET[tf]
        sub = d[["atr_14","close",tr,"is_high_vol"]].dropna().copy()
        sub["rr"] = (sub[tr] * sub["close"]).abs() / sub["atr_14"].replace(0, np.nan)
        clean = sub[sub["rr"].between(0, 10)]

        med = clean["rr"].median()
        mn  = clean["rr"].mean()
        pct_above15 = (clean["rr"] > 1.5).mean() * 100

        ax.hist(clean["rr"], bins=60, color="steelblue", edgecolor="none", alpha=0.8)
        ax.axvline(1.0, color="red",    linestyle="--", linewidth=0.9)
        ax.axvline(med, color="green",  linestyle="-",  linewidth=1.2, label=f"med={med:.2f}")
        ax.set_title(f"{key}", fontsize=8, fontweight="bold")
        ax.set_xlabel("R:R", fontsize=7); ax.tick_params(labelsize=6)
        ax.legend(fontsize=6)

        rr_rows.append({"Dataset":key, "Median R:R":round(med,3),
                        "Mean R:R":round(mn,3), ">1.5 R:R %":round(pct_above15,1)})

plt.suptitle("Risk-to-Reward Distribution (ATR-based SL) — All 12 Datasets",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("tm_rr_distribution.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: tm_rr_distribution.png\n")
print(pd.DataFrame(rr_rows).to_string(index=False))


Saved: tm_rr_distribution.png

Dataset  Median R:R  Mean R:R  >1.5 R:R %
 BTC_1h       0.333     0.477         4.2
 BTC_4h       0.311     0.470         4.8
BTC_12h       0.312     0.475         4.7
 BTC_1d       0.335     0.494         4.5
 ETH_1h       0.339     0.480         4.2
 ETH_4h       0.319     0.473         4.5
ETH_12h       0.322     0.478         4.8
 ETH_1d       0.344     0.497         4.8
 SOL_1h       0.368     0.491         3.7
 SOL_4h       0.356     0.483         4.0
SOL_12h       0.360     0.484         3.3
 SOL_1d       0.376     0.487         3.3


### TM-4. Automatic Signal Risk Score
LOW / MEDIUM / HIGH per bar across all 12 datasets, with win rate per tier.

In [14]:
risk_rows = []
for asset in ASSETS:
    for tf in TIMEFRAMES:
        key = f"{asset}_{tf}"
        d   = datasets[key]
        tc  = TARGET_DIR[tf]
        sub = d[["atr_ratio","dd_from_ath","is_high_vol",tc]].dropna().copy()

        conds = [
            (sub["is_high_vol"]==1) & (sub["atr_ratio"]>1.5) & (sub["dd_from_ath"]<-0.30),
            (sub["is_high_vol"]==0) & (sub["atr_ratio"]<1.0) & (sub["dd_from_ath"]>-0.15),
        ]
        sub["risk"] = np.select(conds, ["HIGH","LOW"], default="MEDIUM")
        sub["tp1"]  = (sub[tc] == 2).astype(int)

        dist = sub["risk"].value_counts(normalize=True) * 100
        wr   = sub.groupby("risk")["tp1"].mean() * 100

        risk_rows.append({
            "Dataset":    key,
            "LOW %":      round(dist.get("LOW",0),1),
            "MED %":      round(dist.get("MEDIUM",0),1),
            "HIGH %":     round(dist.get("HIGH",0),1),
            "WR LOW":     round(wr.get("LOW",float("nan")),1),
            "WR MED":     round(wr.get("MEDIUM",float("nan")),1),
            "WR HIGH":    round(wr.get("HIGH",float("nan")),1),
        })

risk_df = pd.DataFrame(risk_rows)
print(risk_df.to_string(index=False))

# Grouped bar chart — win rate by risk tier for all datasets
fig, ax = plt.subplots(figsize=(18, 5))
x   = np.arange(len(risk_df))
w   = 0.28
ax.bar(x - w, risk_df["WR LOW"],  w, color="#2ecc71", label="LOW",    edgecolor="none")
ax.bar(x,     risk_df["WR MED"],  w, color="#f39c12", label="MEDIUM", edgecolor="none")
ax.bar(x + w, risk_df["WR HIGH"], w, color="#e74c3c", label="HIGH",   edgecolor="none")
ax.axhline(60, color="grey", linestyle="--", linewidth=1, label="60% threshold")
ax.set_xticks(x); ax.set_xticklabels(risk_df["Dataset"], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Win Rate (%)"); ax.set_title("Win Rate by Risk Tier — All 12 Datasets", fontsize=12)
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("tm_risk_score.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nSaved: tm_risk_score.png")


Dataset  LOW %  MED %  HIGH %  WR LOW  WR MED  WR HIGH
 BTC_1h   20.8   79.2       0    13.8    13.7      NaN
 BTC_4h   21.6   78.4       0    27.1    25.7      NaN
BTC_12h   22.3   77.7       0    35.8    35.3      NaN
 BTC_1d   23.1   76.9       0    41.3    40.9      NaN
 ETH_1h    8.0   92.0       0    24.7    18.3      NaN
 ETH_4h    8.6   91.4       0    37.9    30.8      NaN
ETH_12h    8.3   91.7       0    45.3    39.0      NaN
 ETH_1d    9.3   90.7       0    44.3    42.8      NaN
 SOL_1h    8.2   91.8       0    32.4    24.9      NaN
 SOL_4h    8.5   91.5       0    40.4    36.6      NaN
SOL_12h    8.4   91.6       0    46.5    41.8      NaN
 SOL_1d   10.8   89.2       0    44.8    44.4      NaN

Saved: tm_risk_score.png


### TM-5. Multi-Asset × Multi-Timeframe Win Rate Matrix

In [15]:
results = []
for asset in ASSETS:
    for tf in TIMEFRAMES:
        key = f"{asset}_{tf}"
        d   = datasets[key]
        tc  = TARGET_DIR[tf]
        valid = d[d[tc].notna()]
        wr    = (valid[tc] == 2).mean()
        results.append({"Asset": asset, "Timeframe": tf,
                        "Win Rate": round(wr*100, 2), "Bars": len(valid)})

wr_df    = pd.DataFrame(results)
pivot_wr = wr_df.pivot(index="Timeframe", columns="Asset", values="Win Rate").reindex(["1h","4h","12h","1d"])
print(pivot_wr.to_string())

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(pivot_wr, annot=True, fmt=".1f", cmap="RdYlGn", center=50,
            linewidths=0.5, ax=ax, cbar_kws={"label": "Win Rate (%)"})
ax.set_title("Unconditional BUY Win Rate (%) — All Assets × Timeframes", fontsize=12)
plt.tight_layout()
plt.savefig("tm_asset_tf_winrate.png", dpi=150, bbox_inches="tight")
plt.close()
print("\nSaved: tm_asset_tf_winrate.png")
print("\nFull table:")
print(wr_df.to_string(index=False))


Asset        BTC    ETH    SOL
Timeframe                     
1h         13.72  18.83  25.54
4h         26.01  31.40  36.89
12h        35.44  39.52  42.18
1d         40.98  42.95  44.44

Saved: tm_asset_tf_winrate.png

Full table:
Asset Timeframe  Win Rate  Bars
  BTC        1h     13.72 77683
  BTC        4h     26.01 19443
  BTC       12h     35.44  6485
  BTC        1d     40.98  3243
  ETH        1h     18.83 77683
  ETH        4h     31.40 19443
  ETH       12h     39.52  6485
  ETH        1d     42.95  3243
  SOL        1h     25.54 50891
  SOL        4h     36.89 12734
  SOL       12h     42.18  4244
  SOL        1d     44.44  2122
